# Yard map, step by step

From a drone video to a labelled 3D plant map:

1. **Look at the video** and how each frame is conditioned (OSD crop, contrast).
2. **Reconstruct** with the streaming Geometric Context Transformer (GCT): depth and
   camera pose per frame, fused into one coloured point cloud.
3. **Find plants**: the YOLO detector ensemble on the saved frames, boxes lifted to 3D
   and clustered, each plant named growth form → family → genus → species, stopping
   at the first rank it is unsure of.
4. **Assign points to plants** with the Point Transformer.
5. **Export** one self-contained HTML viewer.

`make_viewer.sh` runs the same chain in one go; `yardmap/live.py` runs it on a live
camera. Run the cells in order with the `.venv310` kernel.

In [ ]:
import os, sys, json, subprocess, time
import numpy as np
import cv2
import matplotlib.pyplot as plt

ROOT = os.path.abspath(".")                      # the project folder (this notebook's folder)
sys.path.insert(0, os.path.join(ROOT, "yardmap"))
PY = sys.executable
M = os.path.join(ROOT, "models")

VIDEO = os.path.join(ROOT, "video", "VID_20260913_192111.mp4")
NAME = "notebook_demo"                           # results go to maps/NAME and viewers/NAME.html
GCT = os.path.join(M, "gct.pt")
MAX_FRAMES = 1500                                # 0 = whole video (~50 min for 7 min of video)
ANCHOR_STRIDE = 72 if MAX_FRAMES == 0 or MAX_FRAMES > 3000 else 12

MAP = os.path.join(ROOT, "maps", NAME)
print("video:", VIDEO, "\nweights:", GCT, "\nmap:", MAP)

## 1. The video and frame conditioning

The FPV feed carries an on-screen display and soft analog colour. Each frame is
trimmed (top, bottom, left, right fractions), contrast-equalised with CLAHE, and
frames that are black or washed out by interference are skipped.

In [ ]:
from reconstruct import make_frame_prep, frame_is_usable, iter_video_rgb

CROP = (0.05, 0.14, 0.22, 0.22)
prep = make_frame_prep(clahe=True, margins=CROP)
cap = cv2.VideoCapture(VIDEO)
n, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS)
print(f"{n} frames at {fps:.2f} fps = {n / fps / 60:.1f} min, {int(cap.get(3))}x{int(cap.get(4))}")
fig, ax = plt.subplots(2, 4, figsize=(16, 6))
for k, f in enumerate(np.linspace(0, n - 1, 4).astype(int)):
    cap.set(cv2.CAP_PROP_POS_FRAMES, f)
    ok, bgr = cap.read()
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    ax[0, k].imshow(rgb); ax[0, k].set_title(f"raw frame {f}")
    ax[1, k].imshow(prep(rgb)); ax[1, k].set_title("model input" + ("" if frame_is_usable(prep(rgb), 14, 18) else " (skipped)"))
for a in ax.flat: a.axis("off")
plt.tight_layout()

## 2. Reconstruction

`StreamMapper` takes one frame at a time. The first 12 frames (one every
`ANCHOR_STRIDE` frames, so they span real motion) are processed together and
fix the map's frame and scale; every later frame attends to them plus a sliding
window of recent keyframes through the KV cache. Each frame's confident,
non-sky depth is unprojected and averaged into 5 mm voxels.

In [ ]:
from reconstruct import StreamMapper

mapper = StreamMapper(
    GCT, num_scale_frames=12, anchor_stride=ANCHOR_STRIDE, kv_cache_sliding_window=16,
    keyframe_interval=30, voxel_size=0.005, max_points=4_000_000, downsample_factor=4,
    conf_threshold=1.5, crop_margins=CROP, clahe=True, min_frame_luma=14, min_frame_std=18,
    sky_model=os.path.join(M, "skyseg.onnx"), fuse="average",
    save_artifacts=MAP, artifact_every=30, source=VIDEO, fps=30)

In [ ]:
t0, last = time.time(), None
for src_id, rgb in iter_video_rgb(VIDEO, fps=30, max_frames=MAX_FRAMES):
    r = mapper.push(rgb, src_id)
    if r is not None:
        last = r
        if mapper.frame_idx % 250 == 0:
            print(f"{mapper.frame_idx:5d} frames  {len(mapper.point_map):>9,} points  "
                  f"{mapper.frame_idx / (time.time() - t0):.1f} fps", flush=True)
last = mapper.flush() or last
mapper.save()
print(f"done: {mapper.frame_idx} frames in {time.time() - t0:.0f}s -> {MAP}")

What the model sees in the last frame: its depth and how confident it is (points under 1.5 are dropped).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].imshow(last["rgb"]); ax[0].set_title("frame")
im = ax[1].imshow(np.squeeze(last["depth"]), cmap="turbo"); ax[1].set_title("depth (map units)"); plt.colorbar(im, ax=ax[1])
im = ax[2].imshow(np.squeeze(last["conf"]), cmap="magma", vmax=5); ax[2].set_title("confidence"); plt.colorbar(im, ax=ax[2])
for a in ax: a.axis("off")

The fused cloud from above, with the camera path.

In [ ]:
pts, cols = mapper.snapshot()
poses = np.stack(mapper.poses)
up = -np.median(poses[:, :3, 1], axis=0); up /= np.linalg.norm(up)       # camera 'up' on average
a = np.cross(up, [1, 0, 0]); a = a if np.linalg.norm(a) > 0.1 else np.cross(up, [0, 1, 0]); a /= np.linalg.norm(a)
b = np.cross(up, a)
sel = np.random.default_rng(0).choice(len(pts), min(len(pts), 400_000), replace=False)
plt.figure(figsize=(10, 10))
plt.scatter(pts[sel] @ a, pts[sel] @ b, c=cols[sel] / 255, s=0.2)
plt.plot(poses[:, :3, 3] @ a, poses[:, :3, 3] @ b, "r-", lw=1, label="camera path")
plt.axis("equal"); plt.legend(); plt.title(f"{len(pts):,} points, top view")

## 3. Plants: detect, lift to 3D, name

Runs over every saved frame (one per second). Two YOLO11m detectors (Open Images
plants and trees in the FPV look, and the same detector self-trained on this
yard) with test-time augmentation; boxes are lifted into the cloud with each
frame's depth and pose, clustered into plants, and the best views of each plant
go to the BioCLIP-2 hierarchical classifier.

In [ ]:
yard = os.path.join(ROOT, "yardmap")
cmd = [PY, "detect_plants.py", "--artifacts", MAP, "--detector", "yolo",
       "--det_weights", f"{M}/detector_fpv.pt,{M}/detector_yard.pt", "--det_threshold", "0.10",
       "--det_tta", "--max_box_frac", "0.85", "--cluster_frac", "0.026", "--merge_iou", "0.25",
       "--merge_sep_frac", "0.8", "--classifier", "hier", "--cls_weights", f"{M}/classifier/best.pt",
       "--tau", f"{M}/classifier/tau85.json", "--classify_top_k", "12"]
out = subprocess.run(cmd, cwd=yard, capture_output=True, text=True)
print("\n".join(l for l in out.stdout.splitlines()[-30:]), out.stderr[-2000:] if out.returncode else "")

In [ ]:
import pandas as pd
pj = os.path.join(MAP, "plants.json")
plants = json.load(open(pj))["plants"] if os.path.exists(pj) else []
if not plants:
    print("No plants found - with a short clip the cloud can be too sparse; raise MAX_FRAMES.")
rows = []
for p in plants:
    h = p.get("hierarchy") or {}
    rows.append({"id": p["id"], "named as": p["species"], "level": h.get("level"),
                 "form": (h.get("form") or {}).get("name"), "family": (h.get("family") or {}).get("name"),
                 "genus": (h.get("genus") or {}).get("name"),
                 "top species": (h.get("top_species") or [{}])[0].get("name"),
                 "views": p["views"], "detector": p["detection_score"]})
pd.DataFrame(rows) if rows else None

In [ ]:
show = [p for p in plants if p.get("crop")][:12]
if show:
  fig, ax = plt.subplots(2, 6, figsize=(18, 6))
  for a, p in zip(ax.flat, show):
    a.imshow(cv2.cvtColor(cv2.imread(os.path.join(MAP, "crops", p["crop"])), cv2.COLOR_BGR2RGB))
    a.set_title(f"#{p['id']} {p['species']}", fontsize=9)
  for a in ax.flat: a.axis("off")

## 4. Which points belong to which plant

The promptable Point Transformer looks at the cloud around each plant together
with where its detection boxes fall in every view, and labels each point as
that plant or not.

In [ ]:
out = subprocess.run([PY, "-m", "plant_seg.segment", "--artifacts", MAP,
                      "--weights", f"{M}/plant_seg.pt", "--threshold", "0.4"],
                     cwd=yard, capture_output=True, text=True)
print(out.stdout[-1500:], out.stderr[-1500:] if out.returncode else "")

## 5. The viewer

One HTML file with the cloud, the plants, their photos and names; it opens offline in any browser.

In [ ]:
html = os.path.join(ROOT, "viewers", f"{NAME}.html")
out = subprocess.run([PY, "export_viewer.py", "--artifacts", MAP, "--out", html, "--min_conf", "0.0"],
                     cwd=yard, capture_output=True, text=True)
print(out.stdout[-500:], out.stderr[-1500:] if out.returncode else "")
print("open:", html)